# Smallprint: end-to-end on Colab or Kaggle

Runs the full Track 1 pipeline: **CuratorKIT** data, **AlignTune** QLoRA SFT and DPO, then **AuditKIT** before/after evaluation.

1. Runtime -> change runtime type -> GPU (a T4 works; an L4 or A100 is faster).
2. Accept the Tiny Aya licence at https://huggingface.co/CohereLabs/tiny-aya-water (done for `mukeshkbj`).
3. Add one Colab secret: `HF_TOKEN` (a write token from https://huggingface.co/settings/tokens).
   The teacher (Aya Expanse 32B) and the judge (Qwen3 235B) are both called through the Hugging Face router
   and billed to your HF credits, so no Cohere key is needed.

In [ ]:
import os
# Safe to re-run: absolute paths, clone only once, then pull the latest code.
BASE = "/content" if os.path.isdir("/content") else ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~"))
%cd {BASE}
![ -d smallprint ] || git clone -q https://github.com/mukeshkbj/smallprint.git
%cd {BASE}/smallprint
!git pull -q
!pip -q install -e ".[data,train,eval]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # Colab asks once to grant this notebook access
except ImportError:
    pass  # not Colab: on Kaggle use Add-ons -> Secrets, or export HF_TOKEN yourself
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not found: add it in the Secrets panel (key icon) and turn on Notebook access, then re-run"

HF_USER = "mukeshkbj"                 # must match the username on your hackathon registration
os.environ["SMALLPRINT_HUB__USER"] = HF_USER
NO_TEACHER = False                    # True = template explanations, no API calls at all
SMOKE_TEST = True                     # True = tiny run to check everything works; False = the real run

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

## 0. Check the teacher and judge (a few calls, a fraction of a cent)


In [ ]:
if not NO_TEACHER:
    !python scripts/00_check_providers.py --n 3


## 1. Data (CuratorKIT)

In [ ]:
flags = ("--no-teacher " if NO_TEACHER else "") + ("--limit 300" if SMOKE_TEST else "--push")
!python scripts/01_build_data.py {flags}
!cat data/build_report.json

## 2. Before: zero-shot Tiny Aya Water
Scoring the base model first means you have a baseline even if training runs out of time.

In [ ]:
n = "--max-samples 200" if SMOKE_TEST else ""
!python scripts/05_evaluate.py --skip-final {n}

## 3. SFT (AlignTune, QLoRA)

In [ ]:
steps = "--max-steps 30" if SMOKE_TEST else "--push"
!python scripts/02_train_sft.py {steps}

## 4. DPO pairs (on-policy mistakes plus targeted negatives) and DPO training

In [ ]:
!python scripts/03_build_dpo.py
steps = "--max-steps 20" if SMOKE_TEST else "--push"
!python scripts/04_train_dpo.py {steps}

## 5. After: AuditKIT before/after

In [ ]:
n = "--max-samples 200" if SMOKE_TEST else ""
!python scripts/05_evaluate.py --with-sft --reuse-base {n}
from IPython.display import Markdown, display
display(Markdown(open("results/summary.md").read()))

## 6. Save results to GitHub
Commit `results/*.json`, `results/summary.md` and the AuditKIT comparison files so judges can see them in the repo.